Imports

In [2]:
import os
import json
import cv2
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from matplotlib import pyplot as plt
from collections import Counter

Paths

In [3]:
DATASET_PATH = os.getenv(
    
    "MHCD2022_PATH",
    "/data/UG/Kiranmoy/datasets/Military-Camouflage-MHCD2022"
    )

DATASET_DIR = Path(DATASET_PATH)

Img_Path = os.path.join(DATASET_DIR, "JPEGImages")
Anno_Path = os.path.join(DATASET_DIR, "Annotations")

Split_Path = os.path.join(DATASET_DIR, "ImageSets", "Main")

Output_Dir = DATASET_DIR.parent / "MHCD2022_COCO_Labels"
os.makedirs(Output_Dir, exist_ok=True)



Categories or Classes

In [4]:
Class_MAP = {    
    "person" : 0,
    "military vehicle": 1,
    "tank" : 2,
    "aeroplane": 3,
    "warship" : 4 
}

Read Split Files

In [5]:
def load_split(split_name):

    path = os.path.join(Split_Path, f"{split_name}.txt")
    
    with open(path, "r") as f:
        ids = [
            line.strip()
            for line in f.readlines()
            if line.strip()
        ]
        
    return ids

XML Parser

In [6]:
def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    width = int(root.find("size/width").text)
    height = int(root.find("size/height").text)

    objects = []
    for obj in root.findall("object"):
        cls = obj.find("name").text
        bbox = obj.find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        objects.append({
            "class": cls,
            "bbox": [xmin, ymin, xmax - xmin, ymax - ymin]
        })
    return width, height, objects

Build COCO

In [7]:
def create_coco(split_ids):

    coco = {
       "images" : [],
       "annotations" : [],
       "categories" : []
 }    
    for cls, cid in Class_MAP.items():
        coco["categories"].append({
            "id": cid,
            "name": cls
        })

    image_id =0
    ann_id =0

    for stem in tqdm(split_ids):
        xml_path = os.path.join(Anno_Path, stem + ".xml")
        img_path = os.path.join(Img_Path, stem + ".jpg")

        if not os.path.exists(xml_path) :
            continue
        if not os.path.exists(img_path) :
            continue
        width, height, objects = parse_xml(xml_path)

        coco["images"].append({
            "id": image_id,
            "file_name": stem + ".jpg",
            "width": width,
            "height": height
        })

        for obj in objects:

            if obj["class"] not in Class_MAP:
                continue

            x,y,w,h = obj["bbox"]
            coco["annotations"].append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": Class_MAP[obj["class"]],
                "bbox": [int(x), int(y), int(w), int(h)],
                "area": w*h,
                "iscrowd": 0
            })
            
            ann_id += 1
        image_id += 1
    
    return coco   

Generate COCO for TrainSet

In [8]:
train_ids = load_split("train")

train_coco = create_coco(train_ids)
with open(os.path.join(Output_Dir, "train_coco.json"), "w") as f:
    json.dump(train_coco, f, indent=4)  

  0%|          | 0/2400 [00:00<?, ?it/s]

100%|██████████| 2400/2400 [00:00<00:00, 3683.33it/s]


Generate COCO for TestSet

In [9]:
test_ids = load_split("test")

test_coco = create_coco(test_ids)
with open(os.path.join(Output_Dir, "test_coco.json"), "w") as f:
    json.dump(test_coco, f, indent=4)

100%|██████████| 600/600 [00:00<00:00, 5544.67it/s]


Generate COCO for ValSet

In [10]:
val_ids = load_split("val")

val_coco = create_coco(val_ids)

with open(os.path.join(Output_Dir, "val_coco.json"), "w") as f:
    json.dump(val_coco, f, indent=4)

print("val_coco.json saved")

100%|██████████| 480/480 [00:00<00:00, 14455.64it/s]

val_coco.json saved


Statistics

In [11]:
print("Train Images:", len(train_coco["images"]))
print("Train Annotations:", len(train_coco["annotations"]))
print("Average Objects/Image:", len(train_coco["annotations"])/len(train_coco["images"]))

Train Images: 2400
Train Annotations: 3481
Average Objects/Image: 1.4504166666666667


In [12]:
print("Train Images:", len(train_coco["images"]))
print("Train Annotations:", len(train_coco["annotations"]))

print("Test Images:", len(test_coco["images"]))
print("Test Annotations:", len(test_coco["annotations"]))

print("Val Images:", len(val_coco["images"]))
print("Val Annotations:", len(val_coco["annotations"]))

Train Images: 2400
Train Annotations: 3481
Test Images: 600
Test Annotations: 920
Val Images: 480
Val Annotations: 682


Claases As Per Annotations

In [13]:
# Reverse mapping
id_to_class = {v: k for k, v in Class_MAP.items()}

counter = Counter()

# Train annotations
for ann in train_coco["annotations"]:
    counter[id_to_class[ann["category_id"]]] += 1

# Test annotations
for ann in test_coco["annotations"]:
    counter[id_to_class[ann["category_id"]]] += 1

# DataFrame
class_dist = pd.DataFrame(
    counter.items(),
    columns=["Class", "Count"]
).sort_values("Count", ascending=False)

# Percentage
class_dist["Percentage"] = (
    class_dist["Count"]
    / class_dist["Count"].sum()
    * 100
)

class_dist

,Class,Count,Percentage
0,person,3364,76.437173
2,tank,399,9.066121
3,aeroplane,286,6.498523
1,military vehicle,198,4.498978
4,warship,154,3.499205


In [14]:
import json
import os

for split in ["train", "val", "test"]:
    p = os.path.join(Output_Dir, f"{split}_coco.json")

    with open(p) as f:
        coco = json.load(f)

    print(
        split,
        len(coco["images"]),
        len(coco["annotations"])
    )

train 2400 3481
val 480 682
test 600 920
